# TACTC — Tissue Adjacency Cross-Check of Transcriptomic Clusters

TACTC checks which parts of a spot-level expression clustering (and its PAGA graph) are supported by the tissue. Expression clusters and PAGA are built from **RNA alone** (standard scanpy), a **nuclear-adjacency graph** is built from the **image alone** (segmented nuclei), and the two are crossed only at the end: every spot is marked **core** (part of a contiguous patch of its cluster) or **scattered** (an isolated fragment), and every PAGA edge is marked by whether the clusters' cores touch in the tissue. Same-spot measurements that the clustering did not use (antibody panel, nuclear morphology) then test the marks.

Folder layout (`WORK_DIR`, set in section 0)
* `TACTC.ipynb` : this notebook
* `tactc/` : the modules (a Python package)
* `input/` : input files, see `input/README.md`
  * `input/visium/` : Space Ranger outputs: `filtered_feature_bc_matrix` (`.h5`, folder or `.tar.gz`) and `spatial` (folder or `.tar.gz`)
  * `input/segmentation/` : nucleus segmentation of the section's image: `cells.csv` (cell table) and `mask.tif` (label mask)
* `output/` : results (created by the notebook)

Plan
1. **DAPI cell count per spot + spOT-NMF (reference-free) → cell counts per group per spot** ← implemented
2. **all DAPI cells as graph nodes** · 3. **hexagon expansion and hexagon / circle DAPI ratio** · 4. **scaling the step-1 counts to the hexagons** ← implemented
5. **spot nodes (one node per hexagon: cell counts, expression, morphology, cell-weighted lattice graph)** ← implemented (replaces cell-level type assignment and per-cell RNA)
6. **group-specific node profiles** (local signatures, needed for step 12) · 9. **cross-group correlations** (optional) · 10. **expression clusters and PAGA** (standard scanpy; no spatial information) · 11. **where the expression clusters are supported by the DAPI cell graph**: core (PAGA ∩ DAPI) vs scattered (PAGA only) nodes, PAGA edges whose clusters touch in the tissue, planted controls · 12. **per-group pySCENIC** · 13. **validation: do the PAGA-only parts behave like false positives?** (pre-registered, `docs/VALIDATION.md`) · 14. supplementary checks · 15. optional step-9 checks


Setup: a **GPU runtime is recommended** (spOT-NMF in 1-3 uses PyTorch; it falls back to CPU, much slower). No R is needed.

## 0. Paths / modules
* **Google Colab**: copy (or clone) this repository to Google Drive and set `WORK_DIR` to that folder. A GPU runtime is recommended for spOT-NMF (1-3).
* **Local Jupyter**: run from the repository root; `WORK_DIR` is the current folder.
* Settings and output paths are recorded in `output/pipeline_state.json`, so every section can be continued after a runtime restart (run this section first, then the section's `-R` cell).


In [ ]:
!pip -q install scanpy anndata tifffile zarr psutil

import os, sys, json, importlib
import numpy as np, pandas as pd, matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/TACTC'      # the repository folder on Google Drive
    TMP_DIR  = '/content/tmp'                      # local scratch (faster than Drive)
else:
    WORK_DIR = os.path.abspath('.')                # the repository root
    TMP_DIR  = f'{WORK_DIR}/.tmp'
MODULE_DIR = f'{WORK_DIR}/tactc'
INPUT_DIR  = f'{WORK_DIR}/input'
OUTPUT_DIR = f'{WORK_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
if not os.path.exists(f'{MODULE_DIR}/__init__.py'):
    raise FileNotFoundError(f'{MODULE_DIR}/__init__.py not found. Set WORK_DIR to the folder that contains tactc/.')
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import tactc
from tactc import io_visium as vio, qc as qc2, dapi_counts as dc, programs as pg, cell_counts as cc2
from tactc import cell_graph as cg, hexagons as hx, spot_nodes as sn, node_graphs as ng, cross_groups as cx
from tactc import paga_nodes as pn, grn, paper_checks as pc, regulon_eval as rv, trajectory as tj
from tactc import dapi_support as ds
for m in (vio, qc2, dc, pg, cc2, cg, hx, sn, ng, cx, pn, grn, pc, rv, tj, ds):
    importlib.reload(m)

def first_existing(*paths):
    """First path that exists (inputs may be a file, a folder or an archive)."""
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError('None of these exist:\n  ' + '\n  '.join(paths) + '\nSee input/README.md.')

# settings and paths are recorded so later steps can be continued after a runtime restart
STATE_PATH = f'{OUTPUT_DIR}/pipeline_state.json'
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(**kw):
    STATE.update(kw)
    json.dump(STATE, open(STATE_PATH, 'w'), indent=1, default=str)
print(f'tactc {tactc.__version__} from', os.path.dirname(tactc.__file__), '| outputs →', OUTPUT_DIR)


## 1. Cell counts per group per spot (DAPI + spOT-NMF)
1. **DAPI cell count**: Mesmer cells inside each spot circle (diameter 55 µm, area overlap) = cells in the area where RNA was captured.
2. **spOT-NMF** (reference-free, optimal-transport NMF): each spot's expression as a mixture of K programs learned from this section. No scRNA-seq reference, no R.
3. **Naming**: programs are named from their top genes, marker sets and proteins (`TOPIC_NAMES`); programs with the same name are merged into one group.
4. **Cell counts**: program usages are shares of RNA → corrected to shares of cells (RNA per cell estimated from the DAPI counts) → × DAPI count → integers + 5–95% interval. Step 4 then multiplies the same shares by the hexagon DAPI counts.

### 1-0. spOT-NMF installation (once per runtime)
* `spot-nmf` (PyPI) pins old versions (numpy 1.26, pandas 2.3, anndata 0.9, scanpy 1.9). Installing it normally would **downgrade Colab's numpy / pandas** and break other packages. It is therefore installed with `--no-deps`, and only its missing light dependencies are added (`rbo` also with `--no-deps`, as it pins numpy < 2). Tested this way with numpy 2.5, pandas 3.1, anndata 0.13, scanpy 1.13 and Python 3.13.
* PyTorch comes with Colab. Versions are recorded in `pipeline_state.json`.

In [ ]:
!pip -q install --no-deps spot-nmf rbo
!pip -q install pygam adjustText distinctipy fastcluster gprofiler-official igraph leidenalg

import importlib.metadata as md, torch
importlib.invalidate_caches()
import spotnmf
SPOT_VERSION = md.version('spot-nmf')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
save_state(SPOT_NMF_VERSION=SPOT_VERSION, TORCH_VERSION=md.version('torch'), DEVICE=DEVICE,
           NUMPY_VERSION=md.version('numpy'), PANDAS_VERSION=md.version('pandas'))
print(f"spot-nmf {SPOT_VERSION} | torch {md.version('torch')} on {DEVICE} | numpy {md.version('numpy')} | pandas {md.version('pandas')}")
if DEVICE == 'cpu':
    print('No GPU: spOT-NMF will run on CPU (slow for ~5,600 spots × 2,000 genes). Runtime → Change runtime type → GPU.')

### 1-1. Loading Visium / spot · gene QC
Inputs: `input/visium/filtered_feature_bc_matrix` (`.h5`, folder or `.tar.gz`) and `input/visium/spatial` (folder or `.tar.gz`). QC rules (from visiumif2niche):
* global 3 MAD (flag only, `regional_low`) · local 3 MAD vs ~18 surrounding spots (removed) · high MT (removed) · floor UMI < 500 or genes < 250 (removed)
* genes detected in fewer than 10 spots are removed; spot `total_counts` keeps its original value (used by the RNA-per-cell correction in 1-5)

In [ ]:
VISIUM_DIR     = f'{INPUT_DIR}/visium'
VISIUM_MATRIX  = first_existing(*(f'{VISIUM_DIR}/filtered_feature_bc_matrix{e}' for e in ('.h5', '', '.tar.gz', '.tar')))
VISIUM_SPATIAL = first_existing(*(f'{VISIUM_DIR}/spatial{e}' for e in ('', '.tar.gz', '.tar')))
save_state(VISIUM_MATRIX=VISIUM_MATRIX, VISIUM_SPATIAL=VISIUM_SPATIAL)

vis_raw = vio.load_visium(VISIUM_MATRIX, VISIUM_SPATIAL, work_dir=f'{TMP_DIR}/visium')

QC_DIR     = f'{OUTPUT_DIR}/qc'
NMADS      = 3       # how many MADs from the median count as abnormal
MIN_COUNTS = 500     # minimum UMI per spot
MIN_GENES  = 250     # minimum detected genes per spot
MIN_SPOTS  = 10      # minimum number of spots a gene must be detected in to be kept

QC = qc2.spot_qc(vis_raw, nmads=NMADS, min_counts=MIN_COUNTS, min_genes=MIN_GENES, min_spots=MIN_SPOTS)
QC.save(QC_DIR)
display(QC.summary())
qc2.plot_qc(QC, vis_raw);
vis = QC.apply(vis_raw)      # from here on only spots·genes that passed QC
save_state(QC=dict(nmads=NMADS, min_counts=MIN_COUNTS, min_genes=MIN_GENES, min_spots=MIN_SPOTS,
                   n_spots=int(vis.n_obs), n_genes=int(vis.n_vars)))
print(f'after QC: {vis.n_obs:,} spots × {vis.n_vars:,} genes (originally {vis_raw.n_obs:,} × {vis_raw.n_vars:,})')

### 1-2. DAPI cells per spot (capture circle)
* Uses a nucleus segmentation of the section's image: `input/segmentation/cells.csv` (cell table: index = mask label; columns `area` (px), `y`, `x` (full-resolution px) and optional channel means) and `input/segmentation/mask.tif` (label mask, same pixel grid as the Visium full-resolution image). Any segmentation tool can be used; the example was made with Mesmer (DeepCell). See `input/README.md`.
* A cell on the circle edge counts by the fraction of its area inside the circle (0–1), so the count is fractional. The centre-based integer count is kept for comparison.
* The mask is scanned once (a few minutes); the result is saved in `output/dapi_counts/` and read on later runs (`RECOUNT = True` to redo). If the saved counts do not cover all post-QC spots they are recomputed.

In [ ]:
SEG_DIR   = f'{INPUT_DIR}/segmentation'
CELLS_CSV = f'{SEG_DIR}/cells.csv'        # cell table (index = mask label; columns area, y, x, channels)
MASK_TIF  = f'{SEG_DIR}/mask.tif'         # label mask
DAPI_DIR  = f'{OUTPUT_DIR}/dapi_counts'
RECOUNT   = False

cells = dc.load_cells(CELLS_CSV)
SPOT_CELLS = None
if dc.SpotCells.exists(DAPI_DIR) and not RECOUNT:
    saved = dc.SpotCells.load(DAPI_DIR)
    if vis.obs_names.isin(saved.counts.index).all():
        SPOT_CELLS = saved.aligned(vis.obs_names)
        print('Read the saved DAPI cell counts.')
    else:
        print('Saved DAPI counts do not cover all post-QC spots → recounting.')
if SPOT_CELLS is None:
    MASK = dc.open_mask(MASK_TIF)
    SPOT_CELLS = dc.cells_in_spots(MASK, cells, vis)
    SPOT_CELLS.save(DAPI_DIR)
N_DAPI = SPOT_CELLS.counts.reindex(vis.obs_names).fillna(0.0)
save_state(CELLS_CSV=CELLS_CSV, MASK_TIF=MASK_TIF, DAPI_DIR=DAPI_DIR)

# checks: cells (grey) and spots (colour = DAPI count) should overlap; DAPI count vs UMI
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
sub = cells.sample(min(len(cells), 100_000), random_state=0)
axes[0].scatter(sub.x, sub.y, s=0.2, c='0.6', linewidths=0)
sc_ = axes[0].scatter(vis.obs.x, vis.obs.y, s=3, c=N_DAPI, cmap='viridis', linewidths=0)
axes[0].invert_yaxis(); axes[0].set_aspect('equal'); fig.colorbar(sc_, ax=axes[0], shrink=0.6, label='DAPI cells in spot')
axes[0].set_title('Mesmer cells (grey) vs Visium spots')
axes[1].hist(N_DAPI, bins=50); axes[1].set_xlabel('DAPI cells per spot (area overlap)'); axes[1].set_ylabel('spots')
axes[2].scatter(N_DAPI, vis.obs['total_counts'], s=2)
axes[2].set_xlabel('DAPI cells per spot'); axes[2].set_ylabel('UMI per spot'); axes[2].set_yscale('log')
fig.tight_layout()

### 1-3. spOT-NMF
* Genes: `GENE_MODE = 'hvg'` (top `N_GENES` variable genes, scanpy) or `'overdispersed'` (spOT-NMF's own selection). The OT ground cost is gene × gene, so memory and time grow with `N_GENES`².
* `K`: number of programs (spOT-NMF has no automatic K choice; visiumif2niche used K = 8). `CONSENSUS = True` runs `N_ITER` replicate fits, combines them and reports the cross-replicate stability (useful for comparing K; `N_ITER`× the time).
* `W_REG`, `H_REG`: entropy regularisation of the usages / spectra (larger = smoother, more mixed). In a synthetic test the package defaults (0.01) recovered the groups poorly and 0.1 well, so 0.1 is the default here; check on real data with 1-4 and the protein check in 1-6.
* `USAGE_REFIT = 'nnls'` (default): spOT-NMF's own usages are pulled toward equal shares by the entropy term (in the synthetic test they correlated r 0.99 with the truth but had the wrong scale, which made the cell counts in 1-5 wrong). Each spot is therefore refitted as a non-negative mixture of the learned programs, which gives calibrated RNA shares (error 0.135 → 0.017; cell shares after 1-5 0.128 → 0.030). `'ot'` keeps the raw usages; they are saved as `theta_ot.csv` either way.
* Results are saved in `output/spotnmf/` and reused when the settings and spots are the same (`RERUN_SPOT = True` to refit).

In [ ]:
PROG_DIR    = f'{OUTPUT_DIR}/spotnmf'
RERUN_SPOT  = False
K           = 8
GENE_MODE   = 'hvg'      # 'hvg' | 'overdispersed'
N_GENES     = 2000
W_REG       = 0.1
H_REG       = 0.1
MAX_ITER    = 100
CONSENSUS   = False
N_ITER      = 10
USAGE_REFIT = 'nnls'     # 'nnls' | 'ot'

PROG = pg.run_spotnmf(vis, PROG_DIR, rerun=RERUN_SPOT, k=K, gene_mode=GENE_MODE, n_genes=N_GENES, w=W_REG, h=H_REG,
                      max_iter=MAX_ITER, consensus=CONSENSUS, n_iter=N_ITER, seed=0, dtype='float32',
                      usage_refit=USAGE_REFIT)
display(PROG.theta.describe().T.round(3))
save_state(PROG_DIR=PROG_DIR, K=K, SPOT_PARAMS={k: v for k, v in PROG.params.items() if k not in ('spots_hash', 'genes')})

### 1-4. Naming programs
Programs are expression programs, not cell types: one program can mix types, and a type can be split over several programs.
* `topic_summary`: per program, the mean usage, the spots where it is the largest, the top genes, the 3 best marker sets (Spearman of the usage with the marker-set score across spots; `pg.MARKERS` = glioblastoma sets, replace for other tissues) and the 3 best proteins.
* `TOPIC_NAMES`: program → group name. Programs with the same name are merged (shares summed); unnamed programs keep their name. `DROP_TOPICS`: programs (or names) to leave out; the remaining shares are renormalised.
* Use the names of `PROTEIN_RULES` (1-6) for groups that have a matching protein (e.g. 'T/NK', 'Macrophage/Mono', 'Endothelial', 'Mural'), so they are checked against the protein panel.

In [ ]:
TSUM = pg.topic_summary(PROG, vis)
TSUM.to_csv(f'{PROG_DIR}/program_summary.csv')
display(TSUM)
cc2.plot_maps(PROG.theta, vis, title=f'spOT-NMF program usages (K = {PROG.k})');

In [ ]:
TOPIC_NAMES = {}        # e.g. {'Program 1': 'Microglia', 'Program 2': 'Tumor: MES-like', 'Program 5': 'Microglia'}
DROP_TOPICS = []        # e.g. ['Program 7']

SHARES = pg.name_topics(PROG.theta, TOPIC_NAMES, drop=DROP_TOPICS)
print('groups:', list(SHARES.columns))
save_state(TOPIC_NAMES=TOPIC_NAMES, DROP_TOPICS=DROP_TOPICS, GROUPS=list(SHARES.columns))

### 1-5. Cell counts per group per spot
* Program usages are shares of the spot's **transcripts**: groups with more RNA per cell (e.g. neurons, tumour cells) are overrepresented and RNA-poor cells (e.g. immune cells) underrepresented.
* `RNA_CORRECTION = 'fit'` (visiumif2niche method; no reference needed): relative RNA per cell of each group is estimated from how the DAPI count of a spot relates to its UMI and group shares across all spots (N ≈ Σ share × UMI × c × f, bounded least squares with a ridge toward no correction). `False` = no correction; `{group: relative RNA per cell}` = set by hand.
* Cells per group = DAPI count × share of cells → integers summing to round(DAPI count) (largest remainder).
* `lo` / `hi` = 5–95% over `N_DRAWS` draws (correction refitted on a bootstrap of spots + multinomial split): uncertainty of the correction and the integer split, not of the deconvolution itself.

In [ ]:
COUNT_DIR      = f'{OUTPUT_DIR}/cell_counts'
RNA_CORRECTION = 'fit'      # 'fit' | False | {group: relative RNA per cell}
RIDGE          = 0.1        # larger = weaker correction
N_DRAWS        = 200

CC = cc2.to_cell_counts(SHARES, N_DAPI, vis.obs['total_counts'], rna_correction=RNA_CORRECTION, ridge=RIDGE,
                        min_cells=2, n_draws=N_DRAWS, seed=0)
CC.save(COUNT_DIR)
display(CC.summary())
display(pd.DataFrame(CC.params['rna_factor_5_95'], index=['f 5%', 'f 95%']).T.round(3))
cc2.plot_maps(CC.counts, vis, title='cells per group per spot (DAPI × spOT-NMF)');

### 1-6. Checks · saving one spot AnnData
* **Protein check**: Spearman ρ between each group's share of cells and its matching Visium protein (CytAssist antibody panel, measured apart from RNA; not the IF image, so it works without PCNA or other IF channels). Only groups named as keys of `PROTEIN_RULES` are tested.
  * **Shift null** (`PROTEIN_N_SHIFT` shifts ≥ `PROTEIN_MIN_SHIFT_UM`, as in visiumif2niche 13-3): the protein map is read at the spot nearest to the shifted position. Both maps keep their spatial autocorrelation, so a ρ that only reflects two smooth patterns is not counted. `z_shift`, one-sided `p_shift`; `passes` = ρ ≥ 0.1 and `p_shift` ≤ 0.05.
* **Saved AnnData** `output/step1_spots.h5ad`: the post-QC Visium spots with every step-1 output
  * `obs`: `n_dapi`, `n_dapi_centroid`, `has_shares`, `n_cells_assigned`, `dominant_group`
  * `obsm`: `topic_theta`, `rna_share`, `cell_frac`, `cell_expected`, `cell_counts`, `cell_counts_lo`, `cell_counts_hi` (columns = `uns['step1']['topics']` / `['groups']`; `cc2.frame(adata, key)` returns a DataFrame)
  * `var['in_corpus']`, `varm['topic_beta']` (gene × program, 0 for genes not used by spOT-NMF)
  * `uns['step1']`: programs, groups, program → group, K, parameters, relative RNA per cell

In [ ]:
PROTEIN_RULES = dict(cc2.PROTEIN_RULES)    # group → matching proteins (keys must be your group names)
PROTEIN_N_SHIFT      = 200
PROTEIN_MIN_SHIFT_UM = 1000.0

if 'protein' in vis.obsm:
    PCHK = cc2.protein_check(CC.cell_frac, vis, rules=PROTEIN_RULES, min_rho=0.1, n_shift=PROTEIN_N_SHIFT,
                             min_shift_um=PROTEIN_MIN_SHIFT_UM, seed=0)
    PCHK.to_csv(f'{COUNT_DIR}/protein_check.csv', index=False)
    display(PCHK)
else:
    print('No protein (Antibody Capture) data: protein check skipped.')

STEP1 = cc2.add_to_anndata(vis, CC, lda=PROG, spot_cells=SPOT_CELLS, topic_names=TOPIC_NAMES)
STEP1_H5AD = f'{OUTPUT_DIR}/step1_spots.h5ad'
STEP1.write_h5ad(STEP1_H5AD)
save_state(STEP1_H5AD=STEP1_H5AD, COUNT_DIR=COUNT_DIR, RNA_CORRECTION=RNA_CORRECTION, RIDGE=RIDGE, N_DRAWS=N_DRAWS)
print(f'[save] {STEP1_H5AD}: {STEP1.n_obs:,} spots, obsm {list(STEP1.obsm.keys())}')

### 1-R. (after a runtime restart) Loading the step-1 results
Run section 0, then this cell, to continue without recomputing step 1.

In [ ]:
import anndata as ad
STEP1 = ad.read_h5ad(STATE['STEP1_H5AD'])
PROG = pg.Programs.load(STATE['PROG_DIR'])
CC = cc2.CellCounts.load(STATE['COUNT_DIR'])
SPOT_CELLS = dc.SpotCells.load(STATE['DAPI_DIR']).aligned(STEP1.obs_names)
print(f"step 1: {STEP1.n_obs:,} spots, K = {PROG.k}, groups {list(STEP1.uns['step1']['groups'])}")

## 2–4. Cell graph · hexagon expansion · counts scaled to hexagons
* **2. Cell graph**: every Mesmer (DAPI) cell of the whole image is a node of one spatial graph (not only cells in spots). Saved in a cell-level AnnData (`output/cells_step4.h5ad`): `obsm['spatial']` (µm), `obsp['spatial_connectivities']` / `['spatial_distances']` (squidpy names), `obsm['if_intensity']` (Mesmer channel means). `X` is empty until per-cell RNA is added in step 6.
* **3. Hexagon expansion**: each spot is expanded to its hexagon (up to the midlines to its neighbours, as in visiumif2niche 9-1b). The lattice uses all in-tissue spots before QC, so cells of QC-removed spots are not pulled into neighbouring spots. Each cell gets a **region**:
  * `circle`: ≥ `MIN_CAPTURE_FRAC` of the cell inside the spot's capture circle → **RNA measured**
  * `ring`: in the hexagon, outside the circle → **RNA not measured** (extrapolated from its spot)
  * `qc_removed`, `no_shares` (spot without step-1 shares), `outside` (no in-tissue spot hexagon: tissue edge, off-tissue debris)
* **hex / circle ratio** per spot = DAPI cells in the hexagon / DAPI cells in the circle (the hexagon is ≈ 3.6× the circle area).
* **4. Scaling**: assumption = ring cells have the same composition as the circle cells of their spot. Scaling the step-1 circle counts by the ratio equals hexagon cell number × step-1 cell share, so the counts are built from the actual cell nodes: circle counts (from circle cells) + ring counts (from ring cells) = hexagon counts, and every spot's counts sum exactly to its number of cell nodes (needed for the assignment in step 5).

### 2–4-0. Inputs (also after a runtime restart)
Uses the objects of section 1 if they are in memory; otherwise reads them from `output/` (run section 0 first).

In [ ]:
import anndata as ad
if 'STEP1' not in globals():
    STEP1 = ad.read_h5ad(STATE['STEP1_H5AD'])
if 'vis' not in globals():
    vis = STEP1                      # post-QC spots (obs x, y; uns mpp)
if 'vis_raw' not in globals():       # all in-tissue spots before QC (hexagon lattice)
    vis_raw = vio.load_visium(STATE['VISIUM_MATRIX'], STATE['VISIUM_SPATIAL'], work_dir=f'{TMP_DIR}/visium')
if 'cells' not in globals():
    cells = dc.load_cells(STATE['CELLS_CSV'])
if 'CC' not in globals():
    CC = cc2.CellCounts.load(STATE['COUNT_DIR'])
if 'SPOT_CELLS' not in globals():
    SPOT_CELLS = dc.SpotCells.load(STATE['DAPI_DIR']).aligned(vis.obs_names)
MPP = float(vis.uns['mpp'])
print(f'{len(cells):,} cells, {vis_raw.n_obs:,} in-tissue spots ({vis.n_obs:,} after QC), {MPP:.4f} µm/px')

### 2. Cell graph (all DAPI cells)
* `GRAPH_METHOD = 'delaunay'`: Delaunay triangulation of the cell centres; edges longer than `MAX_EDGE_UM` are removed (adapts to local density and does not bridge empty areas). `'knn'`: `KNN_K` nearest neighbours, also cut at `MAX_EDGE_UM`.
* Check: median degree (~5–6 for Delaunay), isolated cells, number of connected components. The edge-length histogram shows whether `MAX_EDGE_UM` cuts inside the cell-spacing distribution.

In [ ]:
GRAPH_METHOD = 'delaunay'    # 'delaunay' | 'knn'
MAX_EDGE_UM  = 30.0
KNN_K        = 6

XY_UM = cells[['x', 'y']].to_numpy(float) * MPP
CONN, DIST = cg.build_graph(XY_UM, method=GRAPH_METHOD, max_edge_um=MAX_EDGE_UM, k=KNN_K)
GSTATS = cg.graph_stats(CONN)
CELLS_AD = cg.cells_anndata(cells, MPP, CONN, DIST)
CELLS_AD.uns['graph'] = dict(method=GRAPH_METHOD, max_edge_um=MAX_EDGE_UM, k=KNN_K, **GSTATS)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(DIST.data, bins=60); axes[0].axvline(MAX_EDGE_UM, color='k', ls='--', lw=1)
axes[0].set_xlabel('edge length (µm)'); axes[0].set_ylabel('edges (counted twice)')
deg = np.asarray(CONN.sum(axis=1)).ravel()
axes[1].hist(deg, bins=np.arange(deg.max() + 2) - 0.5); axes[1].set_xlabel('degree'); axes[1].set_ylabel('cells')
fig.tight_layout()

### 3. Hexagon expansion · hex / circle ratio
* `MIN_CAPTURE_FRAC`: a cell counts as `circle` (RNA measured) if at least this fraction of its area lies inside the capture circle (from the step-1 mask overlap).
* Outputs in `output/hexagons/`: `cell_hexagons.csv.gz` (cell → spot, region, capture fraction), `spot_hexagons.csv` (per spot: `n_hex`, `n_circle`, `n_ring`, `n_dapi_area` (step 1), `hex_ratio` = n_hex / n_dapi_area, `hex_ratio_members` = n_hex / n_circle).

In [ ]:
HEX_DIR          = f'{OUTPUT_DIR}/hexagons'
MIN_CAPTURE_FRAC = 0.5

HEXA = hx.assign_hexagons(cells, vis_raw.obs, vis.obs_names, SPOT_CELLS,
                          shares_spots=CC.rna_share.dropna().index, min_capture_frac=MIN_CAPTURE_FRAC)
HEXA.save(HEX_DIR)
display(HEXA.summary())
display(HEXA.spots[['n_hex', 'n_circle', 'n_ring', 'n_dapi_area', 'hex_ratio', 'hex_ratio_members']].describe().round(2))
hx.annotate_cells(CELLS_AD, HEXA)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sub = CELLS_AD.obs.sample(min(CELLS_AD.n_obs, 150_000), random_state=0)
colors = {'circle': 'tab:red', 'ring': 'tab:orange', 'qc_removed': 'tab:purple', 'no_shares': 'tab:green', 'outside': '0.75'}
for r, col in colors.items():
    m = sub['region'] == r
    axes[0].scatter(sub.loc[m, 'x_um'], sub.loc[m, 'y_um'], s=0.3, c=col, label=f'{r} ({int((CELLS_AD.obs.region == r).sum()):,})', linewidths=0)
axes[0].invert_yaxis(); axes[0].set_aspect('equal'); axes[0].legend(markerscale=12, fontsize=8, loc='upper right')
axes[0].set_title('cell regions')
axes[1].hist(HEXA.spots['hex_ratio'].dropna().clip(upper=10), bins=50)
axes[1].set_xlabel('hexagon / circle DAPI cells (clipped at 10)'); axes[1].set_ylabel('spots')
fig.tight_layout()
save_state(HEX_DIR=HEX_DIR, MIN_CAPTURE_FRAC=MIN_CAPTURE_FRAC,
           GRAPH=dict(method=GRAPH_METHOD, max_edge_um=MAX_EDGE_UM, k=KNN_K))

### 4. Step-1 counts scaled to the hexagons
* Per spot: circle counts = step-1 share of cells × circle cells, ring counts = share × ring cells (each split into integers by largest remainder), hexagon counts = circle + ring. Spots without step-1 shares get no counts (their cells stay `no_shares`).
* `lo` / `hi`: 5–95% of `N_DRAWS_HEX` multinomial draws of the hexagon's cells (uncertainty of the integer split only).
* Ring counts rest on the assumption above; they are kept separate (`cell_counts_ring`) so later steps can treat measured and extrapolated cells differently.

In [ ]:
N_DRAWS_HEX = 200

HC = hx.scale_counts(HEXA, CC.cell_frac, n_draws=N_DRAWS_HEX, seed=0)
HC.save(HEX_DIR)
display(HC.summary())
cc2.plot_maps(HC.hex, vis, title='cells per group per hexagon (circle + ring)');

### 2–4 saving
* `output/cells_step4.h5ad` (cell level): `obs` cell_id, x_um, y_um, area_um2, spot, region, capture_frac, rna_measured · `obsm` spatial, spatial_px, if_intensity · `obsp` spatial_connectivities, spatial_distances · `uns` graph, step4
* `output/step4_spots.h5ad` (spot level): the step-1 AnnData + `obs` n_hex, n_circle_cells, n_ring_cells, hex_ratio, hex_ratio_members · `obsm` cell_counts_hex, cell_counts_circle, cell_counts_ring, cell_expected_hex, cell_counts_hex_lo / _hi (columns = `uns['step1']['groups']`)

In [ ]:
CELLS_AD.uns['step4'] = dict(min_capture_frac=MIN_CAPTURE_FRAC, hex_spacing_px=HEXA.spacing_px,
                             groups=list(HC.hex.columns), n_draws_hex=N_DRAWS_HEX)
CELLS_H5AD = f'{OUTPUT_DIR}/cells_step4.h5ad'
CELLS_AD.write_h5ad(CELLS_H5AD)
SPOTS4 = hx.add_to_spots(STEP1, HEXA, HC)
STEP4_H5AD = f'{OUTPUT_DIR}/step4_spots.h5ad'
SPOTS4.write_h5ad(STEP4_H5AD)
save_state(CELLS_H5AD=CELLS_H5AD, STEP4_H5AD=STEP4_H5AD, N_DRAWS_HEX=N_DRAWS_HEX)
print(f'[save] {CELLS_H5AD}: {CELLS_AD.n_obs:,} cells, {CELLS_AD.obsp["spatial_connectivities"].nnz // 2:,} edges')
print(f'[save] {STEP4_H5AD}: {SPOTS4.n_obs:,} spots')

### 4-R. (after a runtime restart) Loading the step 2–4 results
Run section 0, then this cell.

In [ ]:
import anndata as ad
CELLS_AD = ad.read_h5ad(STATE['CELLS_H5AD'])
SPOTS4 = ad.read_h5ad(STATE['STEP4_H5AD'])
HEXA = hx.HexAssignment.load(STATE['HEX_DIR'])
HC = hx.HexCounts.load(STATE['HEX_DIR'])
print(f"cells {CELLS_AD.n_obs:,} ({dict(CELLS_AD.obs['region'].value_counts())}), spots {SPOTS4.n_obs:,}")

## 5. Spot nodes (one node per hexagon)
Within a hexagon, Visium RNA is the same for every arrangement of its cells, so which DAPI cell is which type, and how its cells differ, cannot be measured. The later steps therefore use **one node per hexagon (spot centre)**, at the resolution RNA was measured. Cell information enters as node attributes and weights:
* **Node weight** = DAPI cells in the hexagon (`obs['node_weight']`); lattice edges are weighted √(nᵢ nⱼ) / median, so nodes with few cells influence the graph less.
* **Expression**
  * `X` = log1p(counts per 10k): removes the factor shared by all genes of a node (cell number, capture efficiency, hexagon ratio), so distances and correlations reflect *what* is expressed. Use this for graphs, Leiden, PAGA and correlations.
  * `layers['counts']` = measured counts · `layers['hex_scaled']` = counts × hexagon / circle ratio (absolute hexagon totals; for amounts only). Hexagons with fewer than `MIN_CIRCLE` circle cells are flagged `ratio_unstable` and their ratio is capped.
  * `layers['decomp_<i>']` = share of the counts from group i (spot decomposition with spot-derived group signatures, visiumif2niche 10c step 1); sums over groups to `counts`. These shares come from the same measurement, so correlations between groups at one node are partly built in (step 9 needs a decomposition null as well as the shift null).
* **Composition**: `obsm['cell_counts_hex' / '_circle' / '_ring']`, `group_frac`, `program_usage` (step-1 RNA shares).
* **Morphology** (`obsm['morphology']`): cell density (cells / mm²), nuclear area mean / median / CV, mean log intensity of each Mesmer channel.
* **Graph**: `obsp['spatial_connectivities']` / `['spatial_distances']` = hexagon lattice (6 neighbours).
* Saved: `output/nodes_step5.h5ad`.

In [ ]:
import anndata as ad
if 'SPOTS4' not in globals():
    SPOTS4 = ad.read_h5ad(STATE['STEP4_H5AD'])
if 'CELLS_AD' not in globals():
    CELLS_AD = ad.read_h5ad(STATE['CELLS_H5AD'])
if 'HC' not in globals():
    HC = hx.HexCounts.load(STATE['HEX_DIR'])
if 'HEXA' not in globals():
    HEXA = hx.HexAssignment.load(STATE['HEX_DIR'])

NODE_DIR    = f'{OUTPUT_DIR}/nodes'
N_TOP_GENES = 2000
EXTRA_GENES = []         # e.g. ['PCNA', 'MKI67']
MIN_CELLS   = 1          # nodes need ≥ this many DAPI cells in the hexagon
MIN_CIRCLE  = 2          # fewer circle cells → ratio_unstable (ratio capped)

os.makedirs(NODE_DIR, exist_ok=True)
GENES5 = sn.select_genes(SPOTS4, N_TOP_GENES, EXTRA_GENES)
SIG = sn.group_signatures(SPOTS4, HC.circle, GENES5)
SIG.to_csv(f'{NODE_DIR}/group_signatures.csv')
NODES = sn.build_spot_nodes(SPOTS4, HEXA, HC, CELLS_AD, GENES5, signatures=SIG, min_cells=MIN_CELLS, min_circle=MIN_CIRCLE)

# checks: decomposition sums to the measured counts; node weights; morphology
Dsum = sum(NODES.layers[k].toarray() for k in NODES.uns['nodes']['decomp_layers'])
print('decomposition sums to counts:', bool(np.allclose(Dsum, NODES.layers['counts'].toarray(), atol=1e-3)))
display(NODES.obs[['n_hex', 'n_circle', 'n_ring', 'hex_ratio', 'node_weight']].describe().round(2))
display(NODES.obsm['morphology'].describe().T.round(2))
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
xy = NODES.obsm['spatial']
for ax, (v, t) in zip(axes, ((NODES.obs['node_weight'], 'DAPI cells per hexagon (node weight)'),
                             (NODES.obsm['morphology']['area_mean_um2'], 'mean nuclear area (µm²)'),
                             (NODES.obs['dominant_group'].cat.codes, 'dominant group'))):
    im = ax.scatter(xy[:, 0], xy[:, 1], c=v, s=6, cmap='viridis' if 'group' not in t else 'tab10', linewidths=0)
    ax.invert_yaxis(); ax.set_aspect('equal'); ax.set_title(t); ax.axis('off'); fig.colorbar(im, ax=ax, shrink=0.6)
fig.tight_layout()

NODES_H5AD = f'{OUTPUT_DIR}/nodes_step5.h5ad'
NODES.write_h5ad(NODES_H5AD)
save_state(NODE_DIR=NODE_DIR, NODES_H5AD=NODES_H5AD, N_TOP_GENES=N_TOP_GENES, MIN_CELLS=MIN_CELLS, MIN_CIRCLE=MIN_CIRCLE)
print(f'[save] {NODES_H5AD}: {NODES.n_obs:,} nodes × {NODES.n_vars:,} genes, layers {list(NODES.layers.keys())}')

### 5-R. (after a runtime restart) Loading the spot nodes

In [ ]:
import anndata as ad
NODES = ad.read_h5ad(STATE['NODES_H5AD'])
print(f"nodes {NODES.n_obs:,} × genes {NODES.n_vars:,}; groups {list(NODES.uns['nodes']['groups'])}")

## 6. Group-specific node profiles · per-group graphs

> **Optional — not part of the main TACTC analysis** (the main analysis is sections 10–13; criteria: `docs/VALIDATION.md`). Step 6's **local signatures** are needed (step 12 units); the per-group graphs are not used by the main analysis.

* **Why local signatures**: with one signature per group for the whole section, the spot decomposition (step 5) gives every group in a node the same fold change, so group-specific spatial change cannot be seen. For each node, the per-cell expression of every group is re-estimated over its neighbourhood (nodes within `LOCAL_RADIUS_UM`, Gaussian weights `LOCAL_SIGMA_UM`) by ridge-regularised least squares toward the section-wide signature (`LOCAL_RIDGE`). Neighbours differ in composition, so a change is attributed to the group whose abundance follows it (the local-profile idea of visiumif2niche 10c).
* Stored as `layers['local_<i>']` (per-cell UMI of group i; 0 where the node has no cell of group i).
* **Per-group graph** `obsp['graph_<i>']`: lattice edges between nodes that both contain group i, weight = max(0, cosine of the two nodes' standardised log fold changes of group i) × √(cells of i in both) / median — high when the group expresses similarly in neighbouring nodes.

In [ ]:
import anndata as ad
from scipy import sparse
if 'NODES' not in globals():
    NODES = ad.read_h5ad(STATE['NODES_H5AD'])
if 'SPOTS4' not in globals():
    SPOTS4 = ad.read_h5ad(STATE['STEP4_H5AD'])

GRAPH_DIR       = f'{OUTPUT_DIR}/node_graphs'
LOCAL_RADIUS_UM = 250.0     # neighbourhood of the local fit (≈ 2 rings of spots)
LOCAL_SIGMA_UM  = 100.0
LOCAL_RIDGE     = 0.1       # pull toward the section-wide signature (larger = less local)
GROUP_MIN_CELLS = 1         # a node 'contains' a group with ≥ this many cells of it

os.makedirs(GRAPH_DIR, exist_ok=True)
GROUPS_N = list(NODES.uns['nodes']['groups'])
LOCAL = ng.local_signatures(NODES.layers['counts'].toarray(), NODES.obsm['cell_counts_circle'], NODES.obsm['spatial'],
                            NODES.varm['group_signature'], LOCAL_RADIUS_UM, LOCAL_SIGMA_UM, LOCAL_RIDGE)
Ch_ = np.asarray(NODES.obsm['cell_counts_hex'])
for i in range(len(GROUPS_N)):
    NODES.layers[f'local_{i}'] = sparse.csr_matrix(LOCAL[i] * (Ch_[:, [i]] >= GROUP_MIN_CELLS))
for k, v in ng.group_graphs(NODES, LOCAL, GROUP_MIN_CELLS).items():
    NODES.obsp[k] = v
display(pd.DataFrame({g: dict(nodes=int((Ch_[:, i] >= GROUP_MIN_CELLS).sum()),
                              edges=int(NODES.obsp[f'graph_{i}'].nnz // 2),
                              mean_weight=float(NODES.obsp[f'graph_{i}'].data.mean()) if NODES.obsp[f'graph_{i}'].nnz else np.nan)
                      for i, g in enumerate(GROUPS_N)}).T.round(3))
NODES.uns['step6'] = dict(local_radius_um=LOCAL_RADIUS_UM, local_sigma_um=LOCAL_SIGMA_UM, local_ridge=LOCAL_RIDGE,
                          group_min_cells=GROUP_MIN_CELLS)
save_state(GRAPH_DIR=GRAPH_DIR, LOCAL_RADIUS_UM=LOCAL_RADIUS_UM, LOCAL_SIGMA_UM=LOCAL_SIGMA_UM, LOCAL_RIDGE=LOCAL_RIDGE,
           GROUP_MIN_CELLS=GROUP_MIN_CELLS)

## 9. Cross-group correlations (shift and decomposition nulls)

> **Optional — not part of the main TACTC analysis** (the main analysis is sections 10–13; criteria: `docs/VALIDATION.md`).  Skip 9, 9-check and 9-R to run the main analysis; sections 10–13 run without step 9.

Group expression per node = log of the step-6 local per-cell signature. Per gene:
* **overlap** (A, B): Pearson r across nodes containing both groups — co-variation where they coexist.
* **neighbour** (A | B): OLS t of the share of B among the cells of the **neighbouring** nodes (own node excluded) in `log A expression ~ 1 + own composition + neighbour B share` — A's expression next to B beyond its own node composition (contact-dependent signal).
* **Shift null** (`N_SHIFT9` shifts ≥ `MIN_SHIFT_UM`): the B side is read at the node nearest to the shifted position. Controls spatial autocorrelation.
* **Decomposition null** (`N_DECOMP` simulations): counts simulated with no spatial change and no interaction (negative binomial around Σ cells × section-wide signature, real composition and capture factors, per-gene dispersion from the data); local signatures and both statistics recomputed. Controls what the estimation alone produces — A and B are estimated from the same counts, and neighbour composition enters the local fits. The shift null cannot catch this.
* **Calibration**: with few null replicates the null sd is noisy and z comes out too wide. Each pair's z is therefore recentred and rescaled by the median and MAD across genes (`EMPIRICAL_NULL = True`; Efron's empirical null, assuming most genes have no effect). Raw z are kept (`*_zraw_*`). In a synthetic test without any interaction this brought false calls from 12 to 0 (8,000 tests).
* A **call** needs q ≤ `ALPHA` against **both** nulls with the same sign. BH runs over all pairs × genes (`FDR_SCOPE = 'all'`, conservative) or within each pair (`'pair'`).
* Saved as a separate AnnData `output/cross_groups_step9.h5ad`: obs = pairs `A|B` (A = outcome, B = neighbour / partner), var = genes, `X` = neighbour z vs the shift null, layers `overlap_r / _z_shift / _z_decomp / _q_shift / _q_decomp / _call`, `neighbor_t / _z_shift / _z_decomp / _q_shift / _q_decomp / _call`.
* Time: dominated by the decomposition null (each of the `N_DECOMP` = 200 simulations re-runs the local fits for every node), likely well over an hour for ~5,600 nodes on Colab; lower `N_DECOMP` for a first pass.

In [ ]:
CROSS_DIR = f'{OUTPUT_DIR}/cross_groups'
MIN_SHIFT_UM = 1000.0   # shift null: minimum shift (µm)
MIN_NODES = 30          # groups need ≥ this many nodes (pairs: ≥ this many shared nodes for the overlap statistic)
N_SHIFT9  = 200
N_DECOMP  = 200
ALPHA     = 0.05
EMPIRICAL_NULL = True
FDR_SCOPE = 'all'       # 'all' | 'pair'

os.makedirs(CROSS_DIR, exist_ok=True)
CROSS = cx.cross_group_tests(NODES, local=LOCAL if 'LOCAL' in globals() else None, min_cells=GROUP_MIN_CELLS,
                             min_nodes=MIN_NODES, n_shift=N_SHIFT9, min_shift_um=MIN_SHIFT_UM, n_decomp=N_DECOMP,
                             alpha=ALPHA, radius_um=LOCAL_RADIUS_UM, sigma_um=LOCAL_SIGMA_UM, ridge=LOCAL_RIDGE, seed=0,
                             empirical_null=EMPIRICAL_NULL, fdr_scope=FDR_SCOPE)
display(CROSS.obs)

def top_hits(cross, stat='neighbor', n=10):
    rows = []
    for i, pair in enumerate(cross.obs_names):
        call = cross.layers[f'{stat}_call'][i] > 0
        z = cross.layers[f'{stat}_z_decomp'][i]
        for j in np.flatnonzero(call):
            rows.append(dict(pair=pair, gene=cross.var_names[j], z_decomp=float(z[j]),
                             z_shift=float(cross.layers[f'{stat}_z_shift'][i, j]),
                             q_decomp=float(cross.layers[f'{stat}_q_decomp'][i, j]),
                             q_shift=float(cross.layers[f'{stat}_q_shift'][i, j])))
    df = pd.DataFrame(rows)
    return df.reindex(df['z_decomp'].abs().sort_values(ascending=False).index).head(n) if len(df) else df
HITS_NB, HITS_OV = top_hits(CROSS, 'neighbor', 50), top_hits(CROSS, 'overlap', 50)
HITS_NB.to_csv(f'{CROSS_DIR}/neighbor_hits_top50.csv', index=False)
HITS_OV.to_csv(f'{CROSS_DIR}/overlap_hits_top50.csv', index=False)
display(HITS_NB.head(20)); display(HITS_OV.head(20))

CROSS_H5AD = f'{OUTPUT_DIR}/cross_groups_step9.h5ad'
CROSS.write_h5ad(CROSS_H5AD)
NODES9_H5AD = f'{OUTPUT_DIR}/nodes_step9.h5ad'
NODES.write_h5ad(NODES9_H5AD)
save_state(CROSS_DIR=CROSS_DIR, CROSS_H5AD=CROSS_H5AD, NODES9_H5AD=NODES9_H5AD, MIN_NODES=MIN_NODES, N_SHIFT9=N_SHIFT9,
           N_DECOMP=N_DECOMP, ALPHA=ALPHA, EMPIRICAL_NULL=EMPIRICAL_NULL, FDR_SCOPE=FDR_SCOPE, MIN_SHIFT_UM=MIN_SHIFT_UM)
print(f'[save] {CROSS_H5AD}: {CROSS.n_obs} pairs × {CROSS.n_vars:,} genes | {NODES9_H5AD}: nodes with step-6 outputs')

### 9-check. Replication split · pseudo-pair negative control

> **Optional — not part of the main TACTC analysis** (the main analysis is sections 10–13; criteria: `docs/VALIDATION.md`). 

* **Replication** (`RUN_REPLICATION`): the tissue is split along its long axis into two halves, each analysed on its own (local signatures and both nulls). Calls of one half are checked in the other (same sign, p ≤ 0.05 against both nulls) and compared with the chance level. Hit lists should be shown to replicate.
* **Pseudo-pair negative control** (`RUN_NEGATIVE_CONTROL`): the cells of one group (default: the most abundant) are split at random into two pseudo-groups with identical true expression, and step 9 is re-run. Every call between the two pseudo-groups is false by construction. Unlike the decomposition null, this does not rely on the simulation model.
* Each check costs about one more full step-9 run (the replication runs two halves; `MIN_SHIFT_UM` may need to be lower for a half section). Set `N_DECOMP_CHECK` lower for a first pass.

In [ ]:
RUN_REPLICATION      = True
RUN_NEGATIVE_CONTROL = True
CONTROL_GROUP        = None        # None = the most abundant group
N_DECOMP_CHECK       = N_DECOMP
N_SHIFT_CHECK        = N_SHIFT9
MIN_SHIFT_HALF_UM    = MIN_SHIFT_UM / 2

check_kw = dict(min_cells=GROUP_MIN_CELLS, min_nodes=MIN_NODES, n_decomp=N_DECOMP_CHECK, alpha=ALPHA,
                radius_um=LOCAL_RADIUS_UM, sigma_um=LOCAL_SIGMA_UM, ridge=LOCAL_RIDGE, seed=0,
                empirical_null=EMPIRICAL_NULL, fdr_scope=FDR_SCOPE)
if RUN_REPLICATION:
    REP_SUM, REP_HITS, REP_HALVES = cx.replication_split(NODES, n_shift=N_SHIFT_CHECK, min_shift_um=MIN_SHIFT_HALF_UM, **check_kw)
    REP_SUM.to_csv(f'{CROSS_DIR}/replication_summary.csv', index=False)
    REP_HITS.to_csv(f'{CROSS_DIR}/replication_hits.csv', index=False)
    display(REP_SUM.round(3))
if RUN_NEGATIVE_CONTROL:
    NEG_SUM, NEG_RES = cx.pseudo_pair_control(NODES, group=CONTROL_GROUP, n_shift=N_SHIFT_CHECK,
                                              min_shift_um=MIN_SHIFT_UM, **check_kw)
    NEG_SUM.to_csv(f'{CROSS_DIR}/negative_control.csv', index=False)
    display(NEG_SUM.round(3))
save_state(RUN_REPLICATION=RUN_REPLICATION, RUN_NEGATIVE_CONTROL=RUN_NEGATIVE_CONTROL, N_DECOMP_CHECK=N_DECOMP_CHECK)

### 9-R. (after a runtime restart) Loading the step 6 and 9 results

In [ ]:
import anndata as ad
NODES = ad.read_h5ad(STATE['NODES9_H5AD'])
CROSS = ad.read_h5ad(STATE['CROSS_H5AD'])
G_ = len(NODES.uns['nodes']['groups'])
LOCAL = np.stack([NODES.layers[f'local_{i}'].toarray() for i in range(G_)]) if 'local_0' in NODES.layers else None
print(f"nodes {NODES.n_obs:,} | cross pairs {CROSS.n_obs}")

## 10. Expression clusters and PAGA (no spatial information)
The expression side of the comparison. Only expression is used here, so that step 11 can ask, independently, where these clusters are supported by the DAPI cell graph.
* **10-1 Preprocessing** (scanpy tutorials: 3k PBMC clustering, spatial basic analysis): raw counts of the node spots → `normalize_total(1e4)` → `log1p` → `highly_variable_genes` (`N_TOP_GENES_10`, flavor `seurat`) → `scale(max_value=10)`.
* **10-2 Clusters and PAGA** (Wolf et al., Genome Biology 2019): PCA → `neighbors` → `leiden` (flavor `igraph`; resolution searched to give `K_CLUSTERS` clusters) → `paga` → `umap(init_pos='paga')`. No node weights, no spatial features, no pseudotime here. Labels: `PRE.obs['leiden_std']`.
* **10-3 k sensitivity**: split-half ARI and the separation of independent features (proteins, morphology) for k = `K_SENS` (supplement).

In [ ]:
import scanpy as sc
import anndata as ad
if 'SPOTS4' not in globals():
    SPOTS4 = ad.read_h5ad(STATE['STEP4_H5AD'])
if 'NODES' not in globals():
    NODES = ad.read_h5ad(STATE.get('NODES9_H5AD', STATE['NODES_H5AD']))

PAGA_DIR        = f'{OUTPUT_DIR}/paga'
N_TOP_GENES_10  = 2000
N_PCS           = 30
N_NEIGHBORS     = 15
K_CLUSTERS      = 6
PAGA_THRESHOLD  = 0.03
MIN_SHIFT_UM_10 = 1000.0

os.makedirs(PAGA_DIR, exist_ok=True)
PRE = pn.preprocess(SPOTS4, NODES, n_top_genes=N_TOP_GENES_10)                                   # 10-1
pn.branch(PRE, 'std', k=K_CLUSTERS, n_pcs=N_PCS, n_neighbors=N_NEIGHBORS, weight_by_cells=False,
          paga_threshold=PAGA_THRESHOLD, seed=0)                                                  # 10-2
IND = ng.independent_features(NODES, SPOTS4)
print('independent features:', {k: list(v.columns) for k, v in IND.items()})

CATS10 = list(PRE.obs['leiden_std'].cat.categories)
PAGA_C = PRE.uns['paga_std']['connectivities']
PAGA_C = pd.DataFrame(PAGA_C.toarray() if hasattr(PAGA_C, 'toarray') else np.asarray(PAGA_C), index=CATS10, columns=CATS10)
PAGA_C.to_csv(f'{PAGA_DIR}/paga_connectivities_std.csv')
CSUM = pn.cluster_summary(NODES[PRE.obs_names], PRE.obs['leiden_std'].to_numpy())
CSUM.to_csv(f'{PAGA_DIR}/cluster_summary.csv')
display(CSUM.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
PRE.uns['paga'] = PRE.uns['paga_std']
sc.pl.paga(PRE, threshold=PAGA_THRESHOLD, ax=axes[0], show=False, frameon=False, title='PAGA (expression only)')
del PRE.uns['paga']
lab = PRE.obs['leiden_std'].astype(str).to_numpy(); u = PRE.obsm['X_umap_std']; xy = PRE.obsm['spatial']
for c in CATS10:
    m = lab == c
    axes[1].scatter(u[m, 0], u[m, 1], s=4, label=c, linewidths=0)
    axes[2].scatter(xy[m, 0], xy[m, 1], s=6, label=c, linewidths=0)
axes[1].set_title('UMAP (PAGA init)'); axes[1].legend(markerscale=3, fontsize=7)
axes[2].invert_yaxis(); axes[2].set_aspect('equal'); axes[2].axis('off'); axes[2].set_title('clusters in the tissue')
fig.tight_layout()
save_state(PAGA_DIR=PAGA_DIR, N_TOP_GENES_10=N_TOP_GENES_10, K_CLUSTERS=K_CLUSTERS, PAGA_THRESHOLD=PAGA_THRESHOLD,
           MIN_SHIFT_UM_10=MIN_SHIFT_UM_10, N_PCS=N_PCS, N_NEIGHBORS=N_NEIGHBORS)

### 10-3. k sensitivity (supplement)
Split-half ARI (fit on one side of a random straight cut, transfer to the other) and the separation of independent features (z beyond the shift null) for several k, expression only (no node weights). Descriptive; `K_CLUSTERS` is not chosen from it.

In [ ]:
K_SENS  = [4, 5, 6, 7, 8]
N_SPLIT = 10
KSENS = pn.k_sensitivity(PRE, IND, K_SENS, n_pcs=N_PCS, n_neighbors=N_NEIGHBORS, weight_by_cells=False,
                         n_split=N_SPLIT, n_shift=200, min_shift_um=MIN_SHIFT_UM_10, seed=0)
KSENS.to_csv(f'{PAGA_DIR}/k_sensitivity.csv', index=False)
display(KSENS.drop(columns=['ari_values'], errors='ignore').round(3))

PAGA_H5AD = f'{OUTPUT_DIR}/paga_step10.h5ad'
PRE.write_h5ad(PAGA_H5AD)
save_state(PAGA_H5AD=PAGA_H5AD, K_SENS=K_SENS)
print(f'[save] {PAGA_H5AD}')

### 10-R. (after a runtime restart) Loading the step-10 results
Run section 0, then this cell.

In [ ]:
import anndata as ad
PRE = ad.read_h5ad(STATE['PAGA_H5AD'])
CATS10 = list(PRE.obs['leiden_std'].cat.categories)
PAGA_THRESHOLD = STATE.get('PAGA_THRESHOLD', 0.03)
PAGA_C = pd.read_csv(f"{STATE['PAGA_DIR']}/paga_connectivities_std.csv", index_col=0)
PAGA_C.index = PAGA_C.index.astype(str); PAGA_C.columns = PAGA_C.columns.astype(str)
print(f"nodes {PRE.n_obs:,} | clusters {len(CATS10)}")

## 11. Where the expression clusters are supported by the DAPI cell graph (PAGA ∩ DAPI)
Expression (step 10) and cell positions (DAPI graph, step 2) are combined here for the first time. Where they agree, a cluster is supported; where only expression puts nodes together, the nodes are candidate false positives (tested in step 13).
* **11-1 DAPI node graph**: two hexagons are linked when ≥ `MIN_LINK_EDGES` edges of the DAPI cell graph (Delaunay between nuclei, ≤ 30 µm, step 2) join a cell of one to a cell of the other. Gaps without cells (tears, necrosis, the edge) break the link, unlike the plain hexagon lattice.
* **11-2 Core vs scattered**: for each cluster, keep only its nodes and their DAPI links and take the size of the connected piece every node sits in. Null: the same number of nodes placed **at random** (`N_PERM_11` permutations). `p ≤ ALPHA_11` → **core** (PAGA ∩ DAPI: part of a patch larger than chance), otherwise **scattered** (PAGA only). A shift null is not used here: shifting the label map keeps its patches, so it cannot ask whether a cluster forms patches.
* **11-3 Tissue-annotated PAGA** (descriptive): DAPI links between the **core** nodes of two clusters (touching = ≥ `MIN_LINKS` links and ≥ `MIN_SHARE` of the smaller core boundary), and separately the links and nodes that involve **scattered** nodes. Each PAGA edge is marked *supported* (cores touch), *through fragments* (more fragment links than core links), *weak contact* or *no contact*; cores touching without a PAGA edge are marked too. The original classes (*PAGA ∩ DAPI*, *PAGA only*, *DAPI only*) stay in `edge_type`. `boundary_neighbors` lists which clusters the boundary nodes touch.
* **11-4 Planted controls** (known answer): a contiguous patch and a random set of the same size (`PLANT_FRAC` of the nodes) get new labels. The patch must be called core and the random set scattered (≥ 90 % each); otherwise step 13 is not valid.

In [ ]:
import anndata as ad
if 'PRE' not in globals():
    PRE = ad.read_h5ad(STATE['PAGA_H5AD'])
if 'PAGA_THRESHOLD' not in globals():
    PAGA_THRESHOLD = STATE.get('PAGA_THRESHOLD', 0.03)
if 'CELLS_AD' not in globals():
    CELLS_AD = ad.read_h5ad(STATE['CELLS_H5AD'])

SUPPORT_DIR    = f'{OUTPUT_DIR}/dapi_support'
MIN_LINK_EDGES = 1        # DAPI cell edges needed to link two hexagons
N_PERM_11      = 200
ALPHA_11       = 0.05
MIN_LINKS      = 10       # 11-3: core–core links for two clusters to 'touch'
MIN_SHARE      = 0.02     # 11-3: and this share of the smaller core boundary
PLANT_FRAC     = 0.03

os.makedirs(SUPPORT_DIR, exist_ok=True)
LAB11 = PRE.obs['leiden_std'].astype(str).to_numpy()
XY11 = np.asarray(PRE.obsm['spatial'], float)
DAPI_W, DAPI_A = ds.node_dapi_graph(CELLS_AD, PRE.obs_names, min_edges=MIN_LINK_EDGES)          # 11-1
SUP = ds.core_scattered(DAPI_A, LAB11, n_perm=N_PERM_11, alpha=ALPHA_11, seed=0)                 # 11-2
SUP['nodes'].index = PRE.obs_names
SETS11 = SUP['nodes']['set'].to_numpy()
EDGES11 = ds.edge_support(DAPI_A, LAB11, SETS11, paga=PAGA_C, paga_threshold=PAGA_THRESHOLD,
                          min_links=MIN_LINKS, min_share=MIN_SHARE)                                # 11-3
CTRL11 = ds.planted_controls(DAPI_A, LAB11, XY11, frac=PLANT_FRAC, n_perm=N_PERM_11, alpha=ALPHA_11, seed=0)  # 11-4

SUP['nodes'].to_csv(f'{SUPPORT_DIR}/nodes.csv')
SUP['clusters'].to_csv(f'{SUPPORT_DIR}/clusters.csv', index=False)
EDGES11.to_csv(f'{SUPPORT_DIR}/edges.csv', index=False)
BN11 = ds.boundary_neighbors(DAPI_A, LAB11, SETS11)                                              # 11-3 boundary contacts
BN11['by_cluster'].to_csv(f'{SUPPORT_DIR}/boundary_neighbors.csv')
BN11['by_set'].to_csv(f'{SUPPORT_DIR}/boundary_neighbors_by_set.csv')
CTRL11.to_csv(f'{SUPPORT_DIR}/planted_controls.csv', index=False)
display(SUP['clusters'].round(3)); display(EDGES11.round(3)); display(BN11['by_cluster']); display(CTRL11.round(3))
if not CTRL11['passes'].all():
    print('WARNING: a planted control failed — the core / scattered split is not reliable on this section; '
          'do not interpret step 13 (check MIN_LINK_EDGES and the DAPI graph).')

PRE.obs['support_set'] = pd.Categorical(SETS11, categories=['core', 'scattered'])
PRE.obs['component_size'] = SUP['nodes']['component_size'].to_numpy()
PRE.obs['p_component'] = SUP['nodes']['p_component'].to_numpy()
PRE.obs['boundary'] = SUP['nodes']['boundary'].to_numpy()
PRE.obsp['dapi_links'] = DAPI_W

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for c in CATS10:
    m = LAB11 == c
    core = m & (SETS11 == 'core'); sca = m & (SETS11 == 'scattered')
    p_ = axes[0].scatter(XY11[core, 0], XY11[core, 1], s=6, label=c, linewidths=0)
    axes[0].scatter(XY11[sca, 0], XY11[sca, 1], s=14, marker='x', color=p_.get_facecolor()[0], linewidths=0.8)
axes[0].set_title('core (dots) and scattered (crosses) nodes'); axes[0].legend(markerscale=3, fontsize=7)
axes[0].invert_yaxis(); axes[0].set_aspect('equal'); axes[0].axis('off')
cs = SUP['clusters'].set_index('cluster')
axes[1].bar(cs.index.astype(str), cs['core'], label='core'); axes[1].bar(cs.index.astype(str), cs['scattered'], bottom=cs['core'], label='scattered')
axes[1].set_xlabel('cluster'); axes[1].set_ylabel('nodes'); axes[1].legend(); axes[1].set_title('core vs scattered per cluster')
cen = pd.DataFrame(XY11).groupby(LAB11).median()
colors = {'supported': 'tab:green', 'through fragments': 'tab:orange', 'weak contact': 'tab:red',
          'no contact': 'tab:red', 'contact without PAGA edge': 'tab:gray'}
for r in EDGES11.itertuples():
    if r.link_mark in colors:
        a_, b_ = cen.loc[r.c1], cen.loc[r.c2]
        axes[2].plot([a_[0], b_[0]], [a_[1], b_[1]], color=colors[r.link_mark], lw=1 + 6 * min(r.paga_connectivity if np.isfinite(r.paga_connectivity) else 0, 0.5),
                     ls='-' if r.paga_edge else ':')
axes[2].scatter(XY11[:, 0], XY11[:, 1], s=2, color='0.85', linewidths=0)
for c in CATS10:
    axes[2].text(cen.loc[c, 0], cen.loc[c, 1], c, ha='center', va='center', fontsize=10, weight='bold')
axes[2].invert_yaxis(); axes[2].set_aspect('equal'); axes[2].axis('off')
axes[2].set_title('tissue-annotated PAGA: green = cores touch, orange = through fragments, red = weak / none, dotted gray = cores touch without PAGA edge', fontsize=8)
fig.tight_layout()

SUPPORT_H5AD = f'{OUTPUT_DIR}/support_step11.h5ad'
PRE.write_h5ad(SUPPORT_H5AD)
save_state(SUPPORT_DIR=SUPPORT_DIR, SUPPORT_H5AD=SUPPORT_H5AD, MIN_LINK_EDGES=MIN_LINK_EDGES, N_PERM_11=N_PERM_11,
           ALPHA_11=ALPHA_11, MIN_LINKS=MIN_LINKS, MIN_SHARE=MIN_SHARE, PLANT_FRAC=PLANT_FRAC)
print(f'[save] {SUPPORT_H5AD}')

### 11-R. (after a runtime restart) Loading the step-11 results
Run section 0, then this cell.

In [ ]:
import anndata as ad
PRE = ad.read_h5ad(STATE['SUPPORT_H5AD'])
CATS10 = list(PRE.obs['leiden_std'].cat.categories)
LAB11 = PRE.obs['leiden_std'].astype(str).to_numpy()
SETS11 = PRE.obs['support_set'].astype(str).to_numpy()
XY11 = np.asarray(PRE.obsm['spatial'], float)
EDGES11 = pd.read_csv(f"{STATE['SUPPORT_DIR']}/edges.csv", dtype={'c1': str, 'c2': str})
CTRL11 = pd.read_csv(f"{STATE['SUPPORT_DIR']}/planted_controls.csv")
print(f"core {int((SETS11 == 'core').sum()):,} | scattered {int((SETS11 == 'scattered').sum()):,} | "
      f"planted controls pass: {bool(CTRL11['passes'].all())}")

### 11-5. Tissue-annotated PAGA on the saved step-11 results (no permutations)
Run this instead of the whole of section 11 when step 11 was already run with the older `edge_support`. It reads `support_step11.h5ad` (DAPI links, core/scattered sets, PAGA), adds the fragment contacts and link marks (`supported`, `through fragments`, `weak contact`, `no contact`) and the boundary-neighbor tables. CPU, seconds. After a runtime restart: section 0, 11-R, then this cell. The previous `edges.csv` is kept as `edges_v1.csv`.


In [ ]:
import importlib, shutil
importlib.reload(ds)
DAPI_A = (PRE.obsp['dapi_links'] >= STATE.get('MIN_LINK_EDGES', 1)).astype(np.int8).tocsr()
PAGA_C = pd.DataFrame(np.asarray(PRE.uns['paga_std']['connectivities'].toarray() if hasattr(PRE.uns['paga_std']['connectivities'], 'toarray')
                                 else PRE.uns['paga_std']['connectivities']), index=CATS10, columns=CATS10)
old = f"{STATE['SUPPORT_DIR']}/edges.csv"
if os.path.exists(old) and not os.path.exists(f"{STATE['SUPPORT_DIR']}/edges_v1.csv"):
    shutil.copy(old, f"{STATE['SUPPORT_DIR']}/edges_v1.csv")
EDGES11 = ds.edge_support(DAPI_A, LAB11, SETS11, paga=PAGA_C, paga_threshold=STATE.get('PAGA_THRESHOLD', 0.03),
                          min_links=STATE.get('MIN_LINKS', 10), min_share=STATE.get('MIN_SHARE', 0.02))
EDGES11.to_csv(old, index=False)
BN11 = ds.boundary_neighbors(DAPI_A, LAB11, SETS11)
BN11['by_cluster'].to_csv(f"{STATE['SUPPORT_DIR']}/boundary_neighbors.csv")
BN11['by_set'].to_csv(f"{STATE['SUPPORT_DIR']}/boundary_neighbors_by_set.csv")
display(EDGES11[['c1', 'c2', 'paga_connectivity', 'links', 'fragment_links', 'fragment_nodes', 'tissue_contact', 'link_mark', 'edge_type']].round(3))
display(BN11['by_cluster'])


## 12. Per-group GRN (pySCENIC)
**Unit**: RNA is measured per spot, so a per-cell network cannot be identified (cells of one group in one node share one estimate). The unit is **(node × group)**: for each group, every node with ≥ `GRN_MIN_CELLS` cells of it, with the group's local per-cell expression (step-6 local signatures, re-estimated for the GRN genes), normalised per unit (CP10k → log1p). This is the unit visiumif2niche used ((spot × type) profiles, 13-1).
* **Genes**: candidates = `GRN_N_CANDIDATES` most variable genes + all detected TFs (allTFs_hg38); per group, targets = top `N_TOP` by variance (detected in ≥ 5% of units), regulators = top `N_TF` TFs.
* **pySCENIC** (as visiumif2niche 13-1): GRNBoost2 (arboreto settings, joblib-parallel) → cisTarget (hg38 v10 clust ranking databases ±10 kb and −500/+100 bp, NES ≥ 3) → AUCell (top 5%). Checkpoints per group in `output/grn/checkpoints/` (a disconnected runtime resumes). pySCENIC 0.12.1 is patched at run time for NumPy 2 (removed aliases) and pandas ≥ 3 (`groupby().apply` change in module building).
* **Region specificity (base score)**: for each regulon and step-11 region, d = (mean AUC inside − outside) / SD, against the shift null of the region map (`GRN_N_SHIFT`) → `z_region`.
* **Bonus points** (weights fixed before looking at the ranking; every component is reported):
  * `bonus_9` (step 9): the regulon's TF has a call (+1) plus min(−log10 p, 10) of the hypergeometric enrichment of called genes among its targets — calls of the same group (neighbour statistic as outcome, overlap statistic as partner), both nulls.
  * `bonus_11` (step 11): in that region, the group's largest positive z (enrichment, edge density, cross weight) with q ≤ 0.05, capped at 5.
  * `total = z_region + [z_region > 0] · (W9 · bonus_9 + W11 · bonus_11)`: the bonuses only lift a regulon in a region where it is more active than outside, so a region-level bonus cannot raise regulons that are not active there; rows without a base score are not ranked. `rank_total` vs `rank_base` (z_region only) shows what the bonuses moved.
* Saved: `output/grn/` (regulons, AUCell, priorities) and a separate AnnData `output/grn_step12.h5ad` (obs = regulons `group | regulon`, var = regions, `X` = total, layers z_region, d, bonus_11; obs tf, n_targets, n_targets_called, tf_called, partners_9, bonus_9).
* Time: GRNBoost2 dominates (visiumif2niche: about 3 h for 9 groups on Colab).

The regulons do not depend on the clusters or on step 11 (they are built per group on node × group units), so step 13 can locate the same regulons in core and scattered regions. The former 12-3 prioritisation (bonuses from steps 9 and 11) is no longer run.

### 12-0. pySCENIC installation (once per runtime)
Same versions as visiumif2niche (pyscenic 0.12.1, arboreto 0.1.6, ctxcore 0.2.0), with the runtime's pandas / numpy kept (never downgraded).

In [ ]:
import importlib.metadata as md, subprocess, sys
keep = [f'pandas=={md.version("pandas")}', f'numpy=={md.version("numpy")}']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyscenic==0.12.1', 'arboreto==0.1.6', 'ctxcore==0.2.0',
                'setuptools<81', *keep], check=True)
print('installed:', {p: md.version(p) for p in ['pyscenic', 'arboreto', 'ctxcore', 'pandas', 'numpy']})
save_state(PYSCENIC_VERSIONS={p: md.version(p) for p in ['pyscenic', 'arboreto', 'ctxcore']})

### 12-1. Resources · units
cisTarget resources (two ranking databases of ~310 MB each, motif annotations, TF list) are downloaded once into `CISTARGET_DIR`; by default into `resources/cistarget/` under `WORK_DIR`, which can be shared between projects.

In [ ]:
import anndata as ad
if 'SPOTS4' not in globals():
    SPOTS4 = ad.read_h5ad(STATE['STEP4_H5AD'])
if 'NODES' not in globals():
    NODES = ad.read_h5ad(STATE.get('NODES9_H5AD', STATE['NODES_H5AD']))

CISTARGET_DIR    = f'{WORK_DIR}/resources/cistarget'
GRN_DIR          = f'{OUTPUT_DIR}/grn'
GRN_N_CANDIDATES = 4000
GRN_MIN_CELLS    = 1
GRN_MIN_UNITS    = 100

os.makedirs(GRN_DIR, exist_ok=True)
CTX = grn.download_cistarget(CISTARGET_DIR)
TF_ALL = [l.strip() for l in open(CTX['tfs']) if l.strip()]
GRN_GENES = grn.candidate_genes(SPOTS4[NODES.obs_names], TF_ALL, n_candidates=GRN_N_CANDIDATES)
PROFILES, GRN_SIG = grn.group_profiles(SPOTS4, NODES, GRN_GENES, min_cells=GRN_MIN_CELLS, min_units=GRN_MIN_UNITS,
                                       radius_um=STATE.get('LOCAL_RADIUS_UM', 250.0),
                                       sigma_um=STATE.get('LOCAL_SIGMA_UM', 100.0), ridge=STATE.get('LOCAL_RIDGE', 0.1))
save_state(CISTARGET_DIR=CISTARGET_DIR, GRN_DIR=GRN_DIR, GRN_N_CANDIDATES=GRN_N_CANDIDATES, GRN_MIN_UNITS=GRN_MIN_UNITS)

### 12-2. GRNBoost2 → cisTarget → AUCell (per group)

In [ ]:
GRN_GROUPS = None     # e.g. ['Tumor'] (None = every group with ≥ GRN_MIN_UNITS units)
N_TOP      = 3000
N_TF       = 500

GRNR = grn.run_grn(PROFILES, CTX, groups=GRN_GROUPS, n_top=N_TOP, n_tf=N_TF, min_frac=0.05, nes_threshold=3.0,
                   min_targets=10, auc_frac=0.05, n_jobs=-1, n_workers=os.cpu_count() or 2, seed=0, out_dir=GRN_DIR)
GRNR.save(GRN_DIR)
display(GRNR.summary)
save_state(N_TOP=N_TOP, N_TF=N_TF)

### 12-R. (after a runtime restart) Loading the step-12 results
Run section 0, then this cell.

In [ ]:
GRNR = grn.GRNResult.load(STATE['GRN_DIR'])
print(f"{GRNR.regulons[['group', 'regulon']].drop_duplicates().shape[0]:,} regulons in "
      f"{GRNR.regulons['group'].nunique()} groups")

## 13. Validation: do the PAGA-only (scattered) parts behave like false positives?
Criteria fixed in `docs/VALIDATION.md` (2026-11-24) before running it on the example data; do not change the rules after seeing the results. Every test uses information **not used to define** core and scattered (which used only expression labels and the DAPI graph). Outputs in `output/support_validation/`.
* **13-0** The planted controls of 11-4 must pass; otherwise stop.
* **13-1 Expression reproducibility** (primary P1): re-cluster on one side of a random straight cut (`N_CUT_13` cuts, both sides), transfer the labels to the other side (kNN in the fitted PCA space), match them to the full labels (Hungarian); per node: does it keep its cluster? Agreement core − scattered with a bootstrap CI over fits. Also among **cluster-boundary nodes only** (nodes with a DAPI link to another cluster), because scattered nodes are often at boundaries, where any clustering is less stable.
* **13-2 Independent measurements** (primary P2): k-NN prediction of the cluster from Visium proteins and node morphology. Folds hold out whole spatial blocks (`BLOCK_UM_13`) and drop training nodes within `BUFFER_UM_13` of a test node, so that a node's tissue neighbours cannot leak into training. Accuracy core − scattered with a spatial block bootstrap CI, the chance level from permuted labels, and a **shift null** (label map and core/scattered sets shifted together, `N_SHIFT_13` shifts), because smooth features leak across blocks. Reported over all nodes and over boundary nodes only. Random 5-fold CV (the first run) is reported alongside.
* **13-3 Regulons** (primary P3): the step-12 regulons on regions `<cluster>|core` and `<cluster>|scattered` (C1–C3 and the naive test, `regulon_eval.exclusion`); share of **tested** (regulon, region) pairs (finite shift-null z) with |z| ≥ 2, core vs scattered (one-sided Fisher test). Pairs without a z (region too small) are not counted; the first run counted them as |z| < 2 (kept as `fisher_all_pairs_p`). Also the largest p after leaving out one core region.
* **13-4 Trajectory** (secondary): PAGA edges whose cores touch in the tissue (11-3); pseudotime (DPT) on all nodes vs on core nodes only, Spearman on the core nodes. Root: the OPC/NPC-like rule of `trajectory.choose_root`, restricted to core nodes.
* **13-5 Verdict**: the hypothesis is **supported** if at least two of P1–P3 hold (P1: CI lower bound > 0; P2 (spatial blocks, all nodes): CI lower bound > 0, core accuracy above chance and `p_shift` ≤ 0.05; P3: Fisher p on tested pairs < 0.05), **not supported** if none holds, **partly supported** otherwise. If P1 or P2 holds overall but not among boundary nodes, report that the difference may come from boundary position. P3 is also flagged when it depends on one core region.

> **Change log 2026-11-25** (after the first run on the real data; `docs/VALIDATION.md`): P2 moved from random folds to spatial-block folds with a buffer and a shift null; P3 counts tested pairs only. Both fix a defect in how the pre-registered comparison was computed (spatial leakage; untested pairs counted as negatives), shown on synthetic data and the saved real-data pairs. The first-run values are kept in the outputs (`cv = random`, `fisher_all_pairs_p`) and must be reported with the new ones.

In [ ]:
import anndata as ad
for name, key in (('PRE', 'SUPPORT_H5AD'), ('SPOTS4', 'STEP4_H5AD')):
    if name not in globals() or (name == 'PRE' and 'support_set' not in PRE.obs):
        globals()[name] = ad.read_h5ad(STATE[key])
if 'NODES' not in globals():
    NODES = ad.read_h5ad(STATE.get('NODES9_H5AD', STATE['NODES_H5AD']))
VAL_DIR = f'{OUTPUT_DIR}/support_validation'
os.makedirs(VAL_DIR, exist_ok=True)
LAB11 = PRE.obs['leiden_std'].astype(str).to_numpy()
SETS11 = PRE.obs['support_set'].astype(str).to_numpy()
BND11 = PRE.obs['boundary'].to_numpy(bool)
XY11 = np.asarray(PRE.obsm['spatial'], float)
CTRL11 = pd.read_csv(f"{STATE['SUPPORT_DIR']}/planted_controls.csv")
display(CTRL11.round(3))
assert CTRL11['passes'].all(), '13-0: a planted control failed — step 13 is not valid on this section.'
print(f"13-0 passed | core {int((SETS11 == 'core').sum()):,}, scattered {int((SETS11 == 'scattered').sum()):,}, "
      f"boundary nodes {int(BND11.sum()):,}")

### 13-1. Expression reproducibility (P1)

In [ ]:
N_CUT_13 = 10
REP13 = ds.label_reproducibility(PRE, LAB11, SETS11, BND11, k=len(set(LAB11)), n_cut=N_CUT_13,
                                 n_pcs=STATE.get('N_PCS', 30), n_neighbors=STATE.get('N_NEIGHBORS', 15), seed=0)
REP13['summary'].to_csv(f'{VAL_DIR}/13-1_reproducibility.csv', index=False)
REP13['nodes'].to_csv(f'{VAL_DIR}/13-1_node_agreement.csv')
display(REP13['summary'].round(3))

### 13-2. Independent measurements (P2)

In [ ]:
BLOCK_UM_13  = 500.0
BUFFER_UM_13 = 200.0     # training nodes closer than this to a test node are dropped
N_SHIFT_13   = 50        # shift null for P2 (min p = 1/51)
FEATS13 = ng.independent_features(NODES, SPOTS4)
IND13 = ds.independent_agreement(FEATS13, list(PRE.obs_names), LAB11, SETS11, XY11, boundary=BND11,
                                 block_um=BLOCK_UM_13, buffer_um=BUFFER_UM_13, n_shift=N_SHIFT_13,
                                 min_shift_um=STATE.get('MIN_SHIFT_UM_10', 1000.0), seed=0)
IND13['summary'].to_csv(f'{VAL_DIR}/13-2_independent.csv', index=False)
IND13['nodes'].to_csv(f'{VAL_DIR}/13-2_node_correct.csv')
display(IND13['summary'].round(3))


### 13-3. Regulons in core and scattered regions (P3)
Needs step 12 (`GRNR`). Regions are `<cluster>|core` and `<cluster>|scattered`; each is compared with all other units of the group (as in `regulon_eval.exclusion`).

In [ ]:
RERUN_13_3 = False   # False: recompute the P3 summary from the saved pair table (seconds); True: re-run regulon_eval (slow, needs step 12)
PAIRS13 = f'{VAL_DIR}/13-3_regulon_pairs.csv'
if RERUN_13_3 or not os.path.exists(PAIRS13):
    if 'GRNR' not in globals():
        GRNR = grn.GRNResult.load(STATE['GRN_DIR'])
    SUPLAB13 = ds.support_labels(LAB11, SETS11)
    REG13 = ds.regulon_support(GRNR, NODES, list(PRE.obs_names), SUPLAB13, XY11, n_shift=200,
                               min_shift_um=STATE.get('MIN_SHIFT_UM_10', 1000.0), n_maps=10, n_cut=10, alpha=0.05, seed=0)
    REG13['pairs'].to_csv(PAIRS13, index=False)
else:
    REG13 = dict(pairs=pd.read_csv(PAIRS13))
    REG13['summary'], REG13['regions'] = ds.regulon_summary(REG13['pairs'])
    print(f'[13-3] summary recomputed from {PAIRS13}')
REG13['summary'].to_csv(f'{VAL_DIR}/13-3_regulon_summary.csv', index=False)
REG13['regions'].to_csv(f'{VAL_DIR}/13-3_regulon_regions.csv', index=False)
display(REG13['summary'].round(4)); display(REG13['regions'])


### 13-4. Trajectory (secondary)

In [ ]:
EDGES11 = pd.read_csv(f"{STATE['SUPPORT_DIR']}/edges.csv", dtype={'c1': str, 'c2': str})
display(EDGES11.loc[EDGES11['paga_edge'] | EDGES11['touching']].round(3))
SC13 = tj.state_scores(PRE)
ROOT13 = tj.choose_root(PRE, NODES, SC13)
core_mask = SETS11 == 'core'
root_i = ROOT13['root_index']
if not core_mask[root_i]:
    s_ = ROOT13['score'].where(core_mask)
    root_i = int(np.nanargmax(s_.to_numpy()))
    print(f'root moved to the best-scoring core node: {PRE.obs_names[root_i]}')
PT13 = ds.pseudotime_core(PRE, 'std', core_mask, root_i)
pt = pd.DataFrame({'cluster': LAB11, 'set': SETS11, 'pt_all': PT13['full'], 'pt_core': PT13['core_only']}, index=PRE.obs_names)
pt.to_csv(f'{VAL_DIR}/13-4_pseudotime.csv')
order = pt.groupby('cluster')[['pt_all', 'pt_core']].median().sort_values('pt_all')
display(order.round(3))
TRAJ13 = pd.DataFrame([dict(root=PRE.obs_names[root_i], spearman_all_vs_core=PT13['spearman'], nodes_used=PT13['nodes_used'],
                            core_nodes=PT13['core_nodes'], paga_edges=int(EDGES11['paga_edge'].sum()),
                            paga_edges_touching=int((EDGES11['edge_type'] == 'PAGA ∩ DAPI').sum()),
                            paga_only_edges=int((EDGES11['edge_type'] == 'PAGA only').sum()))])
TRAJ13.to_csv(f'{VAL_DIR}/13-4_trajectory.csv', index=False)
display(TRAJ13)

### 13-5. Verdict (pre-registered rule)

In [ ]:
# after a runtime restart: load saved 13-1 / 13-2 / 13-3 summaries that are not in memory
if 'REP13' not in globals():
    REP13 = dict(summary=pd.read_csv(f'{VAL_DIR}/13-1_reproducibility.csv'))
if 'IND13' not in globals():
    IND13 = dict(summary=pd.read_csv(f'{VAL_DIR}/13-2_independent.csv'))
if 'REG13' not in globals():
    REG13 = dict(summary=pd.read_csv(f'{VAL_DIR}/13-3_regulon_summary.csv'))
N_CUT_13 = globals().get('N_CUT_13', STATE.get('N_CUT_13', 10))
r1 = REP13['summary'].set_index('nodes')
r2 = IND13['summary'].set_index(['cv', 'nodes'])
s3 = REG13['summary'].set_index('set')
def _p2(row):
    return bool(row['ci_low'] > 0 and row['core'] > row['chance_core'] and row['p_shift'] <= 0.05)
P1 = bool(r1.loc['all', 'ci_low'] > 0)
P1_boundary = bool(r1.loc['boundary', 'ci_low'] > 0)
P2 = _p2(r2.loc[('spatial_block', 'all')])
P2_boundary = _p2(r2.loc[('spatial_block', 'boundary')]) if ('spatial_block', 'boundary') in r2.index else None
P2_random_folds = bool(r2.loc[('random', 'all'), 'ci_low'] > 0 and r2.loc[('random', 'all'), 'core'] > r2.loc[('random', 'all'), 'chance_core'])
P3 = bool('fisher_tested_p' in s3 and s3['fisher_tested_p'].iloc[0] < 0.05)
P3_loo = bool(P3 and s3['fisher_tested_loo_max_p'].iloc[0] < 0.05)
P3_all_pairs = bool('fisher_all_pairs_p' in s3 and s3['fisher_all_pairs_p'].iloc[0] < 0.05)
n_ok = P1 + P2 + P3
verdict = 'supported' if n_ok >= 2 else ('not supported' if n_ok == 0 else 'partly supported')
notes = []
if P1 and not P1_boundary: notes.append('P1 not among boundary nodes: may come from boundary position')
if P2 and P2_boundary is False: notes.append('P2 not among boundary nodes: may come from boundary position')
if P3 and not P3_loo: notes.append('P3 depends on one core region')
VERDICT13 = pd.DataFrame([dict(P1_reproducibility=P1, P1_boundary_only=P1_boundary, P2_independent=P2,
                               P2_boundary_only=P2_boundary, P3_regulons=P3, P3_leave_one_region_out=P3_loo,
                               primary_met=n_ok, verdict=verdict, note='; '.join(notes),
                               first_run_P2_random_folds=P2_random_folds, first_run_P3_all_pairs=P3_all_pairs)])
VERDICT13.to_csv(f'{VAL_DIR}/13-5_verdict.csv', index=False)
display(VERDICT13.T)
save_state(VAL_DIR=VAL_DIR, N_CUT_13=N_CUT_13, BLOCK_UM_13=BLOCK_UM_13, BUFFER_UM_13=BUFFER_UM_13, N_SHIFT_13=N_SHIFT_13)


### 13-R. (after a runtime restart) Loading the step-13 results

In [ ]:
VAL_DIR = STATE['VAL_DIR']
for f in ('13-1_reproducibility', '13-2_independent', '13-3_regulon_summary', '13-4_trajectory', '13-5_verdict'):
    print(f); display(pd.read_csv(f'{VAL_DIR}/{f}.csv').round(3))

## 14. Supplementary checks
Run at the end on the saved results. Outputs in `output/paper/`.
* **14-1 Composition vs a standard baseline**: pipeline group shares and a scanpy marker score (`sc.tl.score_genes`; marker sets from `pg.MARKERS` via `pc.BASELINE_MAP`) against the matching Visium proteins: ρ of each, shift-null z of each, paired difference with a spatial block bootstrap 95% CI (visiumif2niche 13-4 A). Needs named programs (1-4) to match groups to proteins.
* **14-2 Morphology**: RNA-independent check of composition against node morphology (shift null).
* **14-3 Transition-node candidates** (exploratory): nodes between two clusters along a PAGA edge, tested against a mixture of real spots; outputs in `output/dapi_support/transition/`.
* **14-4 Spot quality** (added after the primary results): quality filter, quality-matched and adjusted core − scattered differences for P1 and P2 with spatial block bootstrap intervals; outputs in `<VAL_DIR>/quality/`.


In [ ]:
import anndata as ad
PAPER_DIR = f'{OUTPUT_DIR}/paper'
BLOCK_UM  = 500.0
N_BOOT    = 500
os.makedirs(PAPER_DIR, exist_ok=True)

# 14-1 composition vs marker-score baseline
if 'STEP1' not in globals():
    STEP1 = ad.read_h5ad(STATE['STEP1_H5AD'])
if 'CC' not in globals():
    CC = cc2.CellCounts.load(STATE['COUNT_DIR'])
if 'protein' in STEP1.obsm:
    MS = pc.marker_scores(STEP1, pg.MARKERS)
    COMP_BASE = pc.composition_vs_baseline(CC.cell_frac, MS, STEP1, dict(cc2.PROTEIN_RULES), n_shift=200,
                                           min_shift_um=STATE.get('MIN_SHIFT_UM', 1000.0), n_boot=N_BOOT, block_um=BLOCK_UM, seed=0)
    COMP_BASE.to_csv(f'{PAPER_DIR}/13-1_composition_vs_marker_score.csv', index=False)
    display(COMP_BASE.round(3))
else:
    print('13-1 skipped: no Visium proteins.')

In [ ]:
# 14-2 RNA-independent morphology check
if 'NODES' not in globals():
    NODES = ad.read_h5ad(STATE.get('NODES9_H5AD', STATE['NODES_H5AD']))
MORPH = pc.morphology_check(NODES, n_shift=200, min_shift_um=STATE.get('MIN_SHIFT_UM', 1000.0), seed=0)
MORPH.to_csv(f'{PAPER_DIR}/13-5_morphology_vs_composition.csv', index=False)
display(MORPH.pivot(index='group', columns='feature', values='z_shift').round(2))
save_state(PAPER_DIR=PAPER_DIR, BLOCK_UM=BLOCK_UM, N_BOOT=N_BOOT)

### 14-3. Transition-node candidates (exploratory, not pre-registered)
Nodes between two expression clusters: mixed spots or a transition state? CPU only, about 5–15 min; needs only the saved step 11 and 13-4 results (`support_step11.h5ad`, `13-4_pseudotime.csv`), so after a runtime restart run section 0 and then this cell.
* **Examined** = expression boundary (≥ 30% of the node's expression-kNN weight on other clusters; partner = the main other cluster) + PAGA edge to the partner + DPT pseudotime between the two clusters' core medians.
* **Mixture null**: the node is rebuilt from real reference spots of the two clusters (counts thinned to its depth, its fitted share) → `p_mix`. Small = not explained as a mixture.
* **Specificity check**: the same test on expression-interior nodes with a one-cluster null. Candidates (BH q ≤ 0.1) are called only if examined nodes reject more often than interior nodes (block bootstrap CI above 0).
* Location (near / middle / far from the partner in the tissue) is reported as context. Spot data cannot place a state in single cells. Design: `tactc/transition.py`.

In [ ]:
# 14-3 transition-node candidates (exploratory)
import anndata as ad, importlib
from tactc import transition as tr
importlib.reload(tr)
TRANS_DIR = f"{STATE['SUPPORT_DIR']}/transition"
os.makedirs(TRANS_DIR, exist_ok=True)
PRE_T = ad.read_h5ad(STATE['SUPPORT_H5AD'])
PT_T = pd.read_csv(f"{STATE['VAL_DIR']}/13-4_pseudotime.csv", index_col=0)['pt_all'].reindex(PRE_T.obs_names).to_numpy(float)
TRANS = tr.transition_nodes(PRE_T, PT_T, tag='std', paga_threshold=STATE.get('PAGA_THRESHOLD', 0.03),
                            min_mix=0.3, near_um=150, far_um=300, n_null=200, n_interior=300, seed=0)
TRANS['nodes'].to_csv(f'{TRANS_DIR}/14-3_nodes.csv')
TRANS['pairs'].to_csv(f'{TRANS_DIR}/14-3_pairs.csv', index=False)
TRANS['test'].to_csv(f'{TRANS_DIR}/14-3_test.csv', index=False)
display(TRANS['test'].T.round(3)); display(TRANS['pairs'].round(3))

N_ = TRANS['nodes']; xy_ = np.asarray(PRE_T.obsm['spatial'], float)
fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].scatter(xy_[:, 0], xy_[:, 1], s=3, color='0.85', linewidths=0)
for L, mk in (('near', 'o'), ('middle', 's'), ('far', '^')):
    m = (N_['examined'] & (N_['location'] == L)).to_numpy()
    ax[0].scatter(xy_[m, 0], xy_[m, 1], s=10, marker=mk, label=f'examined, {L}', linewidths=0)
m = N_['candidate'].to_numpy()
ax[0].scatter(xy_[m, 0], xy_[m, 1], s=40, facecolors='none', edgecolors='k', linewidths=0.8, label='candidate')
ax[0].invert_yaxis(); ax[0].set_aspect('equal'); ax[0].axis('off'); ax[0].legend(fontsize=7, markerscale=2)
ax[0].set_title('examined nodes by tissue location; circles = candidates')
e = N_[N_['examined']]; i_ = N_[N_['interior_check']]
ax[1].hist([i_['mix_residual'].dropna(), e['mix_residual'].dropna()], bins=40, label=['interior', 'examined'], density=True)
ax[1].set_xlabel('relative residual of the mixture fit'); ax[1].legend(); ax[1].set_title('mixture residual')
fig.tight_layout(); fig.savefig(f'{TRANS_DIR}/14-3.png', dpi=150)
save_state(TRANS_DIR=TRANS_DIR)


### 14-4. Spot quality and the core / scattered differences (added after the primary results)
Scattered nodes hold much less RNA and fewer nuclei than core nodes, which could explain P1 and P2 by spot quality alone. This check was added after the primary results (`docs/VALIDATION.md`, change log) and does not change the verdict. CPU, a few minutes; needs only `support_step11.h5ad` and the per-node results of 13-1 and 13-2, so after a runtime restart run section 0 and then this cell.
* **Quality filter** (≥ `MIN_CELLS_Q` nuclei and ≥ `MIN_COUNTS_Q` counts over the clustering genes) crossed with core / scattered.
* **Matched**: each scattered node vs the 3 core nodes of the same cluster (and boundary status) with the most similar log RNA and log nucleus counts (caliper 0.25 SD).
* **Adjusted**: OLS with cluster, boundary, log RNA and log nuclei (all nodes; nodes passing the filter).
* **Intervals**: spatial block bootstrap with 500 µm blocks (primary) and 1,000 µm blocks (sensitivity); point estimates do not depend on the block size.


In [ ]:
# 14-4 spot quality (added after the primary results)
import anndata as ad, importlib
from tactc import quality as qm
importlib.reload(qm)
MIN_CELLS_Q, MIN_COUNTS_Q = 10, 500
BLOCKS_Q = (500.0, 1000.0)
N_BOOT_Q = 2000
QUAL_DIR = f"{STATE['VAL_DIR']}/quality"
os.makedirs(QUAL_DIR, exist_ok=True)
PRE_Q = ad.read_h5ad(STATE['SUPPORT_H5AD'])
QTAB = qm.quality_table(PRE_Q)
QTAB.to_csv(f'{QUAL_DIR}/node_quality.csv')
for f_ in ('13-1_node_agreement.csv', '13-2_node_correct.csv'):
    if not os.path.exists(f"{STATE['VAL_DIR']}/{f_}"):
        raise FileNotFoundError(f"{STATE['VAL_DIR']}/{f_} not found: run 13-1 and 13-2 first (or copy their outputs there).")
OUT_Q = {'P1': pd.read_csv(f"{STATE['VAL_DIR']}/13-1_node_agreement.csv", index_col=0)['agreement'],
         'P2': pd.read_csv(f"{STATE['VAL_DIR']}/13-2_node_correct.csv", index_col=0)['correct_spatial_block']}
QUAL = qm.quality_check(QTAB, OUT_Q, min_cells=MIN_CELLS_Q, min_counts=MIN_COUNTS_Q, block_sizes=BLOCKS_Q,
                        n_boot=N_BOOT_Q, seed=0)
QUAL['summary'].to_csv(f'{QUAL_DIR}/14-4_quality_summary.csv', index=False)
QUAL['crosstab'].to_csv(f'{QUAL_DIR}/14-4_quality_filter_crosstab.csv', index=False)
QUAL['medians'].to_csv(f'{QUAL_DIR}/14-4_quality_medians.csv')
QUAL['medians_by_cluster'].to_csv(f'{QUAL_DIR}/14-4_quality_medians_by_cluster.csv')
display(QUAL['summary'].round(3)); display(QUAL['crosstab'].round(3)); display(QUAL['medians'].round(1))
save_state(QUAL_DIR=QUAL_DIR, MIN_CELLS_Q=MIN_CELLS_Q, MIN_COUNTS_Q=MIN_COUNTS_Q, BLOCKS_Q=list(BLOCKS_Q), N_BOOT_Q=N_BOOT_Q)


### 14-5. Export for the manuscript figures
Collects the files that `TACTC_preprint/scripts/make_figures.py` needs into `<OUTPUT_DIR>/figure_source/`: node coordinates with cluster and core / scattered labels (map panels), the tissue-annotated PAGA table, and the 14-3 and 14-4 tables. CPU, seconds; run section 0 first.


In [ ]:
# 14-5 export for the manuscript figures
import anndata as ad, shutil, glob
FIG_SRC = f'{OUTPUT_DIR}/figure_source'
os.makedirs(FIG_SRC, exist_ok=True)
P_ = ad.read_h5ad(STATE['SUPPORT_H5AD'])
xy_ = np.asarray(P_.obsm['spatial'], float)
pd.DataFrame({'x_um': xy_[:, 0], 'y_um': xy_[:, 1], 'cluster': P_.obs['leiden_std'].astype(str).to_numpy(),
              'set': P_.obs['support_set'].astype(str).to_numpy(), 'boundary': P_.obs['boundary'].to_numpy(bool)},
             index=P_.obs_names).to_csv(f'{FIG_SRC}/nodes_xy.csv')
copied = []
for f_ in [f"{STATE['SUPPORT_DIR']}/edges.csv", f"{STATE['SUPPORT_DIR']}/boundary_neighbors.csv",
           f"{STATE['SUPPORT_DIR']}/boundary_neighbors_by_set.csv"] + \
          glob.glob(f"{STATE['SUPPORT_DIR']}/transition/14-3_*.csv") + glob.glob(f"{STATE['VAL_DIR']}/quality/*.csv"):
    if os.path.exists(f_):
        shutil.copy(f_, FIG_SRC); copied.append(os.path.basename(f_))
print('nodes_xy.csv +', copied, '→', FIG_SRC)


## 15. Optional: step-9 checks
Only if step 9 was run (not part of the main analysis).
* **15-1 Step 9 vs naive testing**: the same statistics with textbook p-values — calls on the real data and on the pseudo-pair negative control.
* **15-2 Planted-effect simulation**: a planted neighbour effect; recall and false calls.

In [ ]:
# 15-1 step 9 vs naive testing (optional: only if step 9 was run)
if 'CROSS' in globals() and CROSS is not None or 'CROSS_H5AD' in STATE:
    if 'CROSS' not in globals():
        CROSS = ad.read_h5ad(STATE['CROSS_H5AD'])
    NG_ = len(CROSS.uns['cross_groups']['groups'])
    NAIVE = pc.naive_calls(CROSS, NG_, alpha=STATE.get('ALPHA', 0.05))
    NAIVE.to_csv(f'{PAPER_DIR}/13-3_naive_vs_pipeline_real.csv')
    display(NAIVE)
    print('real data, total calls — naive:', NAIVE[['neighbor_naive', 'overlap_naive']].sum().to_dict(),
          '| pipeline:', NAIVE[['neighbor_pipeline', 'overlap_pipeline']].sum().to_dict())
    if 'NEG_RES' in globals():
        NAIVE_NEG = pc.naive_calls(NEG_RES, len(NEG_RES.uns['cross_groups']['groups']))
        pseudo = [p for p in NAIVE_NEG.index if 'pseudo' in p and p.count('pseudo') == 2]
        NAIVE_NEG.loc[pseudo].to_csv(f'{PAPER_DIR}/13-3_naive_vs_pipeline_negative_control.csv')
        display(NAIVE_NEG.loc[pseudo])
    else:
        print('pseudo-pair negative control not in memory (run 9-check) — only the real-data comparison is shown.')
else:
    print('13-3 skipped: step 9 (optional) was not run.')

In [ ]:
# 15-2 planted-effect simulation (optional: tests step 9)
RUN_13_4 = 'CROSS_H5AD' in STATE
if RUN_13_4:
    if 'NODES' not in globals():
        NODES = ad.read_h5ad(STATE.get('NODES9_H5AD', STATE['NODES_H5AD']))
    _tot = pd.Series(np.asarray(NODES.obsm['cell_counts_hex']).sum(axis=0), index=NODES.uns['nodes']['groups']).sort_values(ascending=False)
    SIM_A, SIM_B   = _tot.index[0], _tot.index[1]      # outcome group, neighbour group (change if needed)
    SIM_N_GENES    = 20
    SIM_EFFECT     = 1.0                               # log change of A's planted genes per SD of B's neighbour share
    SIM_N_DECOMP   = STATE.get('N_DECOMP', 200)
    SIM_N_SHIFT    = STATE.get('N_SHIFT9', 200)

    SIM, PLANTED = pc.planted_simulation(NODES, SIM_A, SIM_B, n_genes=SIM_N_GENES, effect=SIM_EFFECT, seed=1)
    CROSS_SIM = cx.cross_group_tests(SIM, local=None, min_cells=STATE.get('GROUP_MIN_CELLS', 1), min_nodes=STATE.get('MIN_NODES', 30),
                                     n_shift=SIM_N_SHIFT, min_shift_um=STATE.get('MIN_SHIFT_UM', 1000.0), n_decomp=SIM_N_DECOMP,
                                     alpha=STATE.get('ALPHA', 0.05), radius_um=STATE.get('LOCAL_RADIUS_UM', 250.0),
                                     sigma_um=STATE.get('LOCAL_SIGMA_UM', 100.0), ridge=STATE.get('LOCAL_RIDGE', 0.1), seed=0,
                                     empirical_null=STATE.get('EMPIRICAL_NULL', True), fdr_scope=STATE.get('FDR_SCOPE', 'all'))
    REC = pd.DataFrame([pc.recovery(CROSS_SIM, SIM_A, SIM_B, PLANTED)])
    REC.to_csv(f'{PAPER_DIR}/13-4_planted_recovery.csv', index=False)
    pd.Series(PLANTED, name='gene').to_csv(f'{PAPER_DIR}/13-4_planted_genes.csv', index=False)
    display(REC)
else:
    print('13-4 skipped: step 9 (optional) was not run.')

## License and credits
TACTC is released under the MIT License (`LICENSE`). `tactc/qc.py`, `io_visium.py`, `dapi_counts.py`, the marker sets in `programs.py`, the RNA-per-cell correction in `cell_counts.py`, the hexagon lattice in `hexagons.py`, the spot decomposition in `spot_nodes.py` and the GRNBoost2 / cisTarget / AUCell wrappers in `grn.py` are adapted from the author's visiumif2niche pipeline. spOT-NMF (Abdelkareem et al., bioRxiv 2025, GPL-3.0) and pySCENIC (Aibar et al., Nat Methods 2017; Van de Sande et al., Nat Protoc 2020, GPL-3.0) are installed at run time and are not redistributed; see `THIRD_PARTY_NOTICES.md`.
